**Home Exercise 1 on Machine Translation**
Implement a `sequence2sequence` to **translate English to Vietnamese**. In this exercise, we will sequentially practice the steps to build a machine learning system for the machine translation task using a `seq2seq` model. These steps *include downloading and preprocessing bilingual data, creating training data, building a `seq2seq` model with attention, visualizing attention data, and translating new sentences on real-world data*.

Data: [IWSLT'15 English-Vietnamese](https://www.kaggle.com/datasets/tuannguyenvananh/iwslt15-englishvietnamese) (Train set: train.en and train.vi||| Val set: tst2012.en and tst2012.vi ||| Test set: tst2013.en and tst2013.vi).

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tqdm
import shutil, sys, zipfile
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

from datetime import datetime
import datetime

print(f"The last time this notebook was run is: {datetime.datetime.now().strftime('%H:%M:%S %d/%m/%y')}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


The last time this notebook was run is: 10:58:51 14/12/25
Using device: cuda


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tuannguyenvananh/iwslt15-englishvietnamese")

print("Path to dataset files:", path)

Path to dataset files: /home/dikhang_hcmut/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1


In [3]:
# Helper_functions
def unzip(path, dest, delete=True):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Zip file does not exist: {path}")
    
    if not zipfile.is_zipfile(path):
        raise zipfile.BadZipFile(f"Not a valid zip file: {path}")
    
    with zipfile.ZipFile(path, 'r') as zip_ref:
        zip_ref.extractall(dest)
        print(f"Unzipped into: {dest}")
    if delete:
        os.remove(path)
    else:
        print(f"Do not remove zipfile.")
    
    return dest

def move_path(src_path: str, dest_dir: str):
    if not os.path.exists(src_path):
        raise FileNotFoundError(f"Invalid {src_path}")
    
    os.makedirs(dest_dir, exist_ok=True)
    
    dst_path = os.path.join(dest_dir, os.path.basename(src_path))
    if os.path.exists(dst_path):
        print(f"Destination {dest_dir} is in used.")
        if os.path.isdir(dst_path):
            shutil.rmtree(dst_path)
        else:
            os.remove(dst_path)

    try:
        new_path = shutil.move(src_path, dest_dir)
        return new_path
    except Exception as e:
        if os.path.isdir(src_path):
            shutil.copytree(src_path, dst_path)
            shutil.rmtree(src_path)
            return dst_path
        else:
            raise e

In [4]:
!ls /home/dikhang_hcmut/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1

In [5]:
path = "/home/dikhang_hcmut/.cache/kagglehub/datasets/tuannguyenvananh/iwslt15-englishvietnamese/versions/1"
src_dir = path
filename = "IWSLT'15 en-vi"

data_dir = "./data"
full_path = os.path.join(src_dir, filename)
# file_path = move_path(full_path, data_dir)
files_path = os.path.join(data_dir, filename)
print("Moved to:", files_path)

Moved to: ./data/IWSLT'15 en-vi


In [6]:
!ls data/IWSLT\'15\ en-vi

dict.en-vi.txt		   train.vi.txt    tst2013.en.txt  vocab.vi.txt
luong-manning-iwslt15.pdf  tst2012.en.txt  tst2013.vi.txt
train.en.txt		   tst2012.vi.txt  vocab.en.txt


## Tokenizer

In [7]:
import os
import spacy
from collections import Counter

DATA_DIR = files_path
FILES = {
    'train_src': os.path.join(DATA_DIR, 'train.en.txt'),
    'train_trg': os.path.join(DATA_DIR, 'train.vi.txt'),
    'val_src':   os.path.join(DATA_DIR, 'tst2012.en.txt'),
    'val_trg':   os.path.join(DATA_DIR, 'tst2012.vi.txt'),
    'test_src':  os.path.join(DATA_DIR, 'tst2013.en.txt'),
    'test_trg':  os.path.join(DATA_DIR, 'tst2013.vi.txt')
}

try:
    spacy_en = spacy.load('en_core_web_sm')
except OSError:
    print("Downloading en_core_web_sm...")
    from spacy.cli import download
    download("en_core_web_sm")
    spacy_en = spacy.load('en_core_web_sm')

def tokenize_en(text):
    return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

def tokenize_vi(text):
    return text.lower().strip().split()


In [8]:
class Vocabulary:
    def __init__(self, freq_threshold=2):
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    def build_vocabulary(self, sentence_list, tokenizer):
        frequencies = Counter()
        idx = 4
        for sentence in sentence_list:
            for word in tokenizer(sentence):
                frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1
    
    def numericalize(self, text, tokenizer):
        tokenized_text = tokenizer(text)
        return [self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text]

def read_lines(filepath):
    print(f"Reading: {filepath}")
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Not found: {filepath}")
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]

import yaml
def load_config(config_path = "config.yml"):
    with open(config_path, mode="r") as f:
        config = yaml.safe_load(f)
        
    return config

### Read data

In [9]:
train_en = read_lines(FILES['train_src'])
train_vi = read_lines(FILES['train_trg'])
val_en = read_lines(FILES['val_src'])
val_vi = read_lines(FILES['val_trg'])
test_en = read_lines(FILES['test_src'])
test_vi = read_lines(FILES['test_trg'])

print(f"\n--- Data Statistics ---")
print(f"Train size: {len(train_en)}")
print(f"Val size:   {len(val_en)}")
print(f"Test size:  {len(test_en)}")
print(f"Sample EN:  {train_en[0]}")
print(f"Sample VI:  {train_vi[0]}")

print("\nBuilding Vocabularies...")
vocab_en = Vocabulary(freq_threshold=2)
vocab_en.build_vocabulary(train_en, tokenize_en)

vocab_vi = Vocabulary(freq_threshold=2)
vocab_vi.build_vocabulary(train_vi, tokenize_vi)

print(f"EN Vocab: {len(vocab_en)} | VI Vocab: {len(vocab_vi)}")

Reading: ./data/IWSLT'15 en-vi/train.en.txt
Reading: ./data/IWSLT'15 en-vi/train.vi.txt
Reading: ./data/IWSLT'15 en-vi/tst2012.en.txt
Reading: ./data/IWSLT'15 en-vi/tst2012.vi.txt
Reading: ./data/IWSLT'15 en-vi/tst2013.en.txt
Reading: ./data/IWSLT'15 en-vi/tst2013.vi.txt

--- Data Statistics ---
Train size: 133317
Val size:   1553
Test size:  1268
Sample EN:  Rachel Pike : The science behind a climate headline
Sample VI:  Khoa học đằng sau một tiêu đề về khí hậu

Building Vocabularies...
EN Vocab: 28172 | VI Vocab: 12517


## Build the Data-loader class and Create training data

In [10]:
class NMTDataset(Dataset):
    def __init__(self, src_lines, trg_lines, src_vocab, trg_vocab, src_tok, trg_tok):
        self.src_lines = src_lines
        self.trg_lines = trg_lines
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.src_tok = src_tok
        self.trg_tok = trg_tok
        
    def __len__(self):
        return len(self.src_lines)
    
    def __getitem__(self, index):
        src_indexs = [1] + self.src_vocab.numericalize(self.src_lines[index], self.src_tok) + [2]
        trg_indexs = [1] + self.trg_vocab.numericalize(self.trg_lines[index], self.trg_tok) + [2]
        return torch.tensor(src_indexs), torch.tensor(trg_indexs)
    
def collate_function(batch):
    src_batch, trg_batch = zip(*batch)
    src_batch = pad_sequence(src_batch, padding_value=0, batch_first=False)
    trg_batch = pad_sequence(trg_batch, padding_value=0, batch_first=False)
    return src_batch, trg_batch

### Load config file

In [11]:
cfg = load_config()
print(f"Loaded config: {cfg}")

Loaded config: {'data': {'batch_size': 8, 'max_len': 50}, 'model': {'enc_emb_dim': 128, 'dec_emb_dim': 128, 'enc_hid_dim': 128, 'dec_hid_dim': 128, 'enc_dropout': 0.3, 'dec_dropout': 0.3}, 'training': {'n_epochs': 8, 'teacher_forcing_ratio': 0.3, 'learning_rate': 0.001, 'clip': 1.0}}


In [12]:
BATCH_SIZE = cfg['data']['batch_size']

train_set = NMTDataset(train_en, train_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)
val_set = NMTDataset(val_en, val_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)
test_set = NMTDataset(test_en, test_vi, vocab_en, vocab_vi, tokenize_en, tokenize_vi)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_function)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_function)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_function)

## Build model

### Encoder class, attention class, decoder class

In [13]:
## encoder with GRU
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, enc_hid_dim, bidirectional=True)
        self.fc = nn.Linear(enc_hid_dim*2, dec_hid_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden
    
class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()
        self.attn = nn.Linear((enc_hid_dim * 2) + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)
        
    def forward(self, hidden, encoder_outputs):
        # hidden: [batch, dec_hid_dim]
        # encoder_outputs: [src_len, batch, enc_hid_dim * 2]

        src_len = encoder_outputs.size(0)

        hidden = hidden.unsqueeze(1).expand(-1, src_len, -1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        energy = torch.tanh(
            self.attn(torch.cat((hidden, encoder_outputs), dim=2))
        )

        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)


class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, dropout, attention):
        super().__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU((enc_hid_dim * 2) + emb_dim, dec_hid_dim)
        self.fc_out = nn.Linear((enc_hid_dim * 2) + dec_hid_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs.detach()).unsqueeze(1)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        prediction = self.fc_out(torch.cat((output.squeeze(0), weighted.squeeze(0), embedded.squeeze(0)), dim=1))
        return prediction, hidden.squeeze(0), a.squeeze(1)

### Seq2Seq class

In [14]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        encoder_outputs, hidden = self.encoder(src)
        encoder_outputs = encoder_outputs.detach()

        input = trg[0]  # <sos>

        outputs = []

        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(
                input, hidden, encoder_outputs
            )

            outputs.append(output)

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1

        # shape: [(trg_len-1)*batch, vocab]
        return torch.cat(outputs, dim=0)


## Training the model

### Configure training parameters

In [15]:
INPUT_DIM = len(vocab_en)
OUTPUT_DIM = len(vocab_vi)
# Hidden Dimensions
ENC_HID_DIM = cfg['model']['enc_hid_dim']
DEC_HID_DIM = cfg['model']['dec_hid_dim']
# Embedding Dimensions
ENC_EMB_DIM = cfg['model']['enc_emb_dim']
DEC_EMB_DIM = cfg['model']['dec_emb_dim']
# Dropout
ENC_DROPOUT = cfg['model']['enc_dropout']
DEC_DROPOUT = cfg['model']['dec_dropout']

LEARNING_RATE = cfg['training']['learning_rate']

attn = Attention(enc_hid_dim=ENC_HID_DIM, dec_hid_dim=DEC_HID_DIM)

enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, ENC_DROPOUT)

dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, DEC_DROPOUT, attn)

model = Seq2Seq(enc, dec, device).to(device)

def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)
            
model.apply(init_weights)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=vocab_vi.stoi["<pad>"])

scaler = torch.amp.GradScaler("cuda")

def train(model, iterator, optimizer, criterion, clip, epoch):
    model.train()
    epoch_loss = 0

    for src, trg in iterator:
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        tf_ratio = max(
            0.1,
            cfg['training']['teacher_forcing_ratio'] * (1 - epoch / N_EPOCHS)
        )

        with torch.amp.autocast('cuda'):
            output = model(src, trg, tf_ratio)

            trg_gold = trg[1:].reshape(-1)
            loss = criterion(output, trg_gold)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, trg in iterator:
            src, trg = src.to(device), trg.to(device)

            output = model(src, trg, 0)

            trg_gold = trg[1:].reshape(-1)
            loss = criterion(output, trg_gold)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


### Training loop

In [16]:
import time
import json
import math

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

N_EPOCHS = cfg['training']['n_epochs']
CLIP = cfg['training']['clip']
SAVE_DIR = 'models'
os.makedirs(SAVE_DIR, exist_ok=True)

best_valid_loss = float('inf')

history = {
    'train_loss': [],
    'valid_loss': [],
    'train_ppl': [],
    'valid_ppl': []
}

print(f"Starting training for {N_EPOCHS} epochs...")
print(f"{'Epoch':^5} | {'Time':^10} | {'Train Loss':^12} | {'Val Loss':^12} | {'Train PPL':^12} | {'Val PPL':^12}")
print("-" * 80)

for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = train(
        model,
        train_loader,
        optimizer,
        criterion,
        CLIP,
        epoch
    )

    valid_loss = evaluate(model, val_loader, criterion)
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    # --- Checkpoint: Lưu model tốt nhất ---
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, 'best_model.pt'))
    
    # --- Tính Metrics ---
    train_ppl = math.exp(train_loss)
    valid_ppl = math.exp(valid_loss)
    
    # --- Lưu History ---
    history['train_loss'].append(train_loss)
    history['valid_loss'].append(valid_loss)
    history['train_ppl'].append(train_ppl)
    history['valid_ppl'].append(valid_ppl)
    
    print(f"{epoch+1:^5} | {epoch_mins}m {epoch_secs}s | {train_loss:^12.3f} | {valid_loss:^12.3f} | {train_ppl:^12.3f} | {valid_ppl:^12.3f}")

with open(os.path.join(SAVE_DIR, 'history.json'), 'w') as f:
    json.dump(history, f)

print("\nTraining Completed. Best model saved as 'models/best_model.pt'.")

Starting training for 8 epochs...
Epoch |    Time    |  Train Loss  |   Val Loss   |  Train PPL   |   Val PPL   
--------------------------------------------------------------------------------


RuntimeError: CUDA driver error: out of memory

### Visualize training results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_comprehensive_training_history(history):
    """
    1. Loss & Perplexity (Train vs Val)
    2. Loss và PPL kết hợp trên cùng một biểu đồ
    """
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # -------------------------------------------------------------
    # Hình 1: Loss và Perplexity riêng biệt (Báo cáo tiêu chuẩn)
    # -------------------------------------------------------------
    
    fig, ax = plt.subplots(1, 2, figsize=(16, 6))
    plt.suptitle('Training and Validation Metrics', fontsize=16)

    # Biểu đồ 1a: Loss
    ax[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
    ax[0].plot(epochs, history['valid_loss'], 'r-o', label='Validation Loss', markersize=4)
    ax[0].set_title('Cross-Entropy Loss vs. Epochs', fontsize=14)
    ax[0].set_xlabel('Epochs', fontsize=12)
    ax[0].set_ylabel('Loss Value', fontsize=12)
    ax[0].legend(loc='upper right')
    ax[0].grid(axis='y', linestyle='--')
    ax[0].set_xticks(epochs)
    ax[0].tick_params(axis='both', which='major', labelsize=10)

    # Biểu đồ 1b: Perplexity (Lưu ý: PPL thường là log scale)
    ax[1].plot(epochs, history['train_ppl'], 'b-o', label='Train PPL', markersize=4)
    ax[1].plot(epochs, history['valid_ppl'], 'r-o', label='Validation PPL', markersize=4)
    ax[1].set_title('Perplexity vs. Epochs (Lower is Better)', fontsize=14)
    ax[1].set_xlabel('Epochs', fontsize=12)
    ax[1].set_ylabel('Perplexity (PPL)', fontsize=12)
    ax[1].set_yscale('log') # Thường dùng log scale cho PPL
    ax[1].legend(loc='upper right')
    ax[1].grid(axis='y', linestyle='--')
    ax[1].set_xticks(epochs)
    ax[1].tick_params(axis='both', which='major', labelsize=10)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()
    
    # -------------------------------------------------------------
    # Hình 2: Loss và PPL Validation trên cùng một trục (Phân tích)
    # -------------------------------------------------------------

    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Vẽ Validation Loss trên trục chính (ax1)
    color = 'tab:blue'
    ax1.set_xlabel('Epochs', fontsize=12)
    ax1.set_ylabel('Validation Loss', color=color, fontsize=12)
    ax1.plot(epochs, history['valid_loss'], color=color, linestyle='-', marker='s', label='Validation Loss')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(axis='y', linestyle='--')
    ax1.set_xticks(epochs)

    # Tạo trục phụ (ax2) cho Validation PPL
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Validation PPL (Log Scale)', color=color, fontsize=12) 
    ax2.plot(epochs, history['valid_ppl'], color=color, linestyle='--', marker='o', label='Validation PPL')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.set_yscale('log')

    # Title và Legend
    plt.title('Validation Performance: Loss vs. Perplexity', fontsize=14)
    
    # Kết hợp Legend từ cả hai trục
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax2.legend(lines + lines2, labels + labels2, loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Gọi hàm
plot_comprehensive_training_history(history)
#

: 

: 

: 

## Translate new sentences and evaluate the model

In [ ]:
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt

def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()
    
    if isinstance(sentence, str):
        tokens = tokenize_en(sentence)
    else:
        tokens = [token.lower() for token in sentence]

    text_to_indices = [src_vocab.stoi.get(token, src_vocab.stoi["<unk>"]) for token in tokens]
    src_tensor = torch.LongTensor([1] + text_to_indices + [2]).unsqueeze(1).to(device) # [src_len, 1]
    
    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src_tensor)
        
    trg_indices = [1] # <sos>
    
    attentions = torch.zeros(max_len, 1, len(src_tensor)).to(device)
    
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indices[-1]]).to(device)
        
        with torch.no_grad():
            output, hidden, attention = model.decoder(trg_tensor, hidden, encoder_outputs)
            
        attentions[i] = attention
        pred_token = output.argmax(1).item()
        trg_indices.append(pred_token)
        
        if pred_token == 2: # <eos>
            break
    
    trg_tokens = [trg_vocab.itos[i] for i in trg_indices]
    return trg_tokens[1:], attentions[:len(trg_tokens)-1]

def display_attention(sentence, translation, attention):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)
    attention = attention.squeeze(1).cpu().detach().numpy()
    cax = ax.matshow(attention, cmap='bone')
    ax.tick_params(labelsize=12)
    
    x_ticks_labels = ['<sos>'] + [t.lower() for t in tokenize_en(sentence)] + ['<eos>']
    y_ticks_labels = translation
    
    ax.set_xticklabels(x_ticks_labels, rotation=45)
    ax.set_yticklabels(y_ticks_labels)
    
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
    
    plt.show()
    plt.close()

In [ ]:
minodel.load_state_dict(torch.load(os.path.join('models', 'best_model.pt')))

import random
idx = random.choice(range(len(test_en)))

src = test_en[idx]
trg = test_vi[idx]

print(f'Src: {src}')
print(f'Trg: {trg}')

# Dịch
translation, attention = translate_sentence(src, vocab_en, vocab_vi, model, device)
pred_sentence = " ".join(translation[:-1])

print(f'Pred: {pred_sentence}')

# Vẽ Attention Map
display_attention(src, translation, attention)

In [ ]:
import sacrebleu

def calculate_bleu(data_src, data_trg, src_vocab, trg_vocab, model, device):
    model.eval()
    preds = []
    targets = []
    
    print("Calculating BLEU Score on Test Set...")
    for i in tqdm(range(len(data_src))):
        src = data_src[i]
        trg = data_trg[i]
        
        pred_tokens, _ = translate_sentence(src, src_vocab, trg_vocab, model, device)
        pred_str = " ".join(pred_tokens[:-1]) 
        
        preds.append(pred_str)
        targets.append(trg)
        
    bleu = sacrebleu.corpus_bleu(preds, [targets])
    return bleu.score

score = calculate_bleu(test_en, test_vi, vocab_en, vocab_vi, model, device)
print(f'BLEU score = {score:.2f}')